# 00 — Samples overview

**Purpose.** Sanity-check the deterministic samples produced by `src.sampling` (task T1). Show counts, temporal pattern frequencies, and random example questions for both datasets, then verify the committed sample files re-derive byte-identically.

**Inputs**: `data/samples/hotpot_1000.json`, `data/samples/musique_500.json`, raw datasets in `data/`.

**Outputs**: display only (no files written).

**Env**: reads `RANDOM_SEED` from `.env` via `src.sampling`.

In [ ]:
import json
import os
import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    raise RuntimeError(f"project root not found from {start}")


ROOT = find_root(Path.cwd())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

print(f"Project root: {ROOT}")


## HotpotQA — temporal vs non-temporal counts

In [ ]:
HOTPOT = json.loads((ROOT / "data/samples/hotpot_1000.json").read_text())
print(f"HotpotQA sample: {len(HOTPOT)} questions")

counts = pd.Series([r["temporal"] for r in HOTPOT]).value_counts()
counts.index = ["temporal" if x else "non-temporal" for x in counts.index]
counts.to_frame("count")


## HotpotQA — temporal pattern frequency (in temporal subset)

In [ ]:
from collections import Counter

temporal_n = sum(1 for r in HOTPOT if r["temporal"])
pat_counts = Counter(p for r in HOTPOT for p in r["patterns"])
df = pd.DataFrame(pat_counts.most_common(), columns=["pattern", "count"])
df["pct_of_temporal"] = (df["count"] / temporal_n * 100).round(1)
df


## MuSiQue — hop-count distribution

In [ ]:
MUSIQUE = json.loads((ROOT / "data/samples/musique_500.json").read_text())
print(f"MuSiQue sample: {len(MUSIQUE)} questions")

hop_counts = pd.Series([r["hop_count"] for r in MUSIQUE]).value_counts().sort_index()
hop_counts.to_frame("count")


## MuSiQue — temporal pattern frequency

In [ ]:
pat_counts_m = Counter(p for r in MUSIQUE for p in r["patterns"])
pd.DataFrame(pat_counts_m.most_common(), columns=["pattern", "count"])


## Random example questions (seeded)

In [ ]:
import random

rng = random.Random(42)

def show(records, label, n=5):
    temp = [r for r in records if r["temporal"]]
    non_temp = [r for r in records if not r["temporal"]]
    print(f"=== {label}: {n} temporal ===")
    for r in rng.sample(temp, min(n, len(temp))):
        print(f"  [{r[\"id\"]}] {r[\"question\"]}")
        if r["patterns"]:
            print(f"    patterns: {\", \".join(r[\"patterns\"])}")
    print(f"\n=== {label}: {n} non-temporal ===")
    for r in rng.sample(non_temp, min(n, len(non_temp))):
        print(f"  [{r[\"id\"]}] {r[\"question\"]}")
    print()

show(HOTPOT, "HotpotQA", n=5)
show(MUSIQUE, "MuSiQue", n=5)


## SHA-stability sanity check

Re-runs `src.sampling.sample_hotpot()` and `sample_musique()` against the raw data and confirms the SHA256 matches the committed sample files. If this drifts, the deterministic guarantee from T1 is broken.

In [ ]:
import hashlib
import tempfile

from src.sampling import sample_hotpot, sample_musique

EXPECTED = {
    "hotpot": "5dcdb24e4152fc956a1809dbb19830b472156c542e127c92bfc83ee3cd0cc598",
    "musique": "bff1ac8a57b7fb1565a3231334792a608684d9138a61fc6ac70b99331a26da19",
}

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    sample_hotpot("data/hotpot_dev_distractor_v1.json", td / "h.json")
    sample_musique("data/musique_ans_v1.0_dev.jsonl", td / "m.json")
    h_h = hashlib.sha256((td / "h.json").read_bytes()).hexdigest()
    h_m = hashlib.sha256((td / "m.json").read_bytes()).hexdigest()

rows = [
    ("hotpot",  h_h, EXPECTED["hotpot"],  h_h == EXPECTED["hotpot"]),
    ("musique", h_m, EXPECTED["musique"], h_m == EXPECTED["musique"]),
]
pd.DataFrame(rows, columns=["dataset", "re_sample_sha256", "committed_sha256", "match"])
